In [ ]:
import ast
import pickle
import pandas as pd
class_info = 'class_info_v1.pkl'
class_list = 'class_list_v1.pkl'
class_relations = 'class_relations_v1.pkl'
class_uri = 'class_uri_v1.pkl'
instance_dict = 'instance_dict_v1.pkl'
instance_dict_emb = 'instance_dict_emb_v1.pkl'
instance_embeddings_all = 'instance_embeddings_all_v1.pkl'

with open(class_info, 'rb') as file:
    class_info = pickle.load(file)
with open(class_list, 'rb') as file:
    class_list = pickle.load(file)
with open(class_relations, 'rb') as file:
    class_relations = pickle.load(file)
with open(class_uri, 'rb') as file:
    class_uri = pickle.load(file)
with open(instance_dict, 'rb') as file:
    instance_dict = pickle.load(file)
with open(instance_dict_emb, 'rb') as file:
    instance_dict_emb = pickle.load(file)
with open(instance_embeddings_all, 'rb') as file:
    instance_embeddings_all = pickle.load(file)


In [ ]:
import os
import ast
import json
import openai
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage,HumanMessage,ToolMessage,BaseMessage
from SPARQLWrapper import SPARQLWrapper, JSON
from typing_extensions import Annotated, TypedDict
from typing import List, Optional, Dict, Callable, Literal, Optional, Sequence
from langchain_core.messages import AIMessage, BaseMessage, SystemMessage, ToolMessage, AnyMessage
from pandas import DataFrame
import operator
from langgraph.graph.message import add_messages
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent, InjectedState, ToolNode, tools_condition
from pydantic import BaseModel
from langgraph.types import Command
from langgraph.graph import StateGraph, MessagesState, START, END
from pydantic import BaseModel, Field
import backoff  

os.environ["OPENAI_API_KEY"] = ""  # Replace with your API key

llm = ChatOpenAI(model_name="o1")

sparql_endpoint = "https://keng.istc.cnr.it/hacid-cs-kg/sparql"
sparql = SPARQLWrapper(sparql_endpoint)


In [ ]:
class InfoExtractionState(TypedDict):
    messages: list
    query: list
    project_description: str
    reasoning_clues: list
    identified_class: list
    relevant_instances: list
    selected_instances: list
    subgraphs: list
    responses: list

# def class_select(state: InfoExtractionState):
def class_select(class_info, query, reasoning_clues):
    """Function to identify relevant classes from a given class dictionary based on user queries."""

    llm4 = ChatOpenAI(model="o3")
    example_output = {'Variable': 'https://w3id.org/hacid/onto/data/Variable'}
    messages = [
        SystemMessage(content=f"""Your are a class identifier. \
                      Given a dictionary which contains class names and their explanations, select the classes which are relevant to user queries. \
                      Follow the tips below: \
                        1. Query: the query is a particular question related to climate services {query}. \
                        2. Context: Use the base answer as the relevant context {reasoning_clues} \
                        3. Reasoning: Do some reasoning to understand the topic, the question, and the lilely answers \
                        4. Class selection: Select the classes which are relevant to the user query and the sub-tasks. \
                      **The maximum number of selected classes is 5.** \
                      The output format should be **a dictionary** with the class name as key and the class URI as value, as shown in the example: {example_output}. \
                      Return only the dictionary containing the class name and its uri in the output. No additional information. \
                      Class dictionary : ```{class_info}``` \
                      Class uri : ```{class_uri}``` \
                      """),
        HumanMessage(content=[{"type":"text", "text":f"Query : {query}"}])]

    response = llm4.invoke(messages)
    class_dict = ast.literal_eval(response.content)
    print(f"Class selection response: {class_dict}")
    # return {"identified_class": class_dict}
    return class_dict

# def class_review(state: InfoExtractionState):
def class_review(class_relations, identified_class, class_info, class_uri, query):
    """Function to review the identified classes and add additional classes if needed."""

    llm5 = ChatOpenAI(model="o3")
    example_output = {'Variable': 'https://w3id.org/hacid/onto/data/Variable'}
    example_output2 = {'Additional Class': 'Not Necessary'}
    example_output3 = {'Additional Class': 'Not Found'}

    messages = [
        SystemMessage(content=f"""Your are an expert in climate services. \
                      Your responsibility is to review the user query and the pre-identified class information {identified_class}, then think about information gap if additional information is required to answer the user query. \
                      You have an access to the information pool {class_relations} which contains the extra information for the pre-identified classes, namely sub-calss, super-class, and relations with other classes. \
                      According to the information gap, you further identify relevant classes which can help answer the user query from the {class_relations}. \
                      You should follow the tips below: \
                        1. Review the user query and the pre-identified class information. \
                        2. Think about the information gap \
                        3. Read the information pool \
                        4. Identify the relevant classes from {class_relations} which can help answer the user query \
                        5. Only select the classes which are absolutely useful or necessary to answer the user query. \
                        6. Do not include redundant information which are already included by the pre-identified classes {identified_class}. \
                        6. The output should be a dictionary with the class name as key and the class URI as value, as shown in the example: {example_output}. \
                        7. If the pre-identified classes are already enough to answer the user query, return {example_output2}. \
                        8. If no additional classes are found, return {example_output3}. \
                        9. Return only the dictionary containing the class name and its uri in the output. No additional information. \
                      Information pool : ```{class_relations}``` \
                      Pre-identified class : ```{identified_class}``` \
                      Class dictionary : ```{class_info}``` \
                      Class uri : ```{class_uri}``` \
                      """),
        HumanMessage(content=[{"type":"text", "text":f"Query : {query}"}])]

    response = llm5.invoke(messages)
    additional_class = ast.literal_eval(response.content)
    # state["additional_class"] = additional_class
    print(f"Class review response: {additional_class}")
    # return {"additional_class": additional_class}
    return additional_class

# @tool
# def class_identifier_tool(class_info:dict, class_relations:dict, class_uri:dict, query:list, tasks:list):
def class_identifier_tool(state: InfoExtractionState):
    """Tool to identify relevant classes according to a user query."""
    
    query = state["query"][-1]
    reasoning_clues = state["reasoning_clues"]

    identified_class = class_select(class_info, query, reasoning_clues)
    # identified_class = class_select(state)
    class_sel_name = list(identified_class.keys())
    print(f"Identified class: {class_sel_name}")

    return {"identified_class": identified_class}


class_graph = StateGraph(InfoExtractionState)
class_graph.add_node("class_tools", class_identifier_tool)
class_graph.set_entry_point("class_tools")
class_graph.set_finish_point("class_tools")
app2 = class_graph.compile()


In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

def class_match(class_name: list, query_class: str) -> list:
    llm6 = ChatOpenAI(model="gpt-4o-mini")

    messages = [
        SystemMessage(content=f"""Your are a class matcher. \
                      Given a class name, you'll find the most similar class from the list of class names {class_name} 
                      Return only one most similar class. No additional information. \
                      Class dictionary : ```{class_name}``` \
                      """),
        HumanMessage(content=[{"type":"text", "text":f"Query : {query_class}"}])]

    response = llm6.invoke(messages)
    return response.content


def cosine_similarity(vec1, vec2):
    """Calculate the cosine similarity between two vectors."""
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    similarity = dot_product / (norm_a * norm_b)
    return similarity


def calculate_similarity(array_emb, user_query_embedding, top_k):
    """
    Calculate cosine similarity between user query embedding and embeddings in DataFrame.

    Parameters:
    df (pd.DataFrame): DataFrame containing embeddings in a column named 'embedding'.
    user_query_embedding (np.ndarray): Embedding of the user query.

    Returns:
    pd.Series: Series containing cosine similarities for each row in the DataFrame.
    """
    
    similarities = []
    for row in array_emb:
        similarity = cosine_similarity(user_query_embedding, row)
        similarities.append(similarity)

    df = pd.DataFrame({'similarity': similarities})
    
    # Return top three most similar embeddings
    return df.nlargest(top_k, 'similarity')

def embed_query(user_query):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    # model = SentenceTransformer('all-distilroberta-v1')
    query_embedding = model.encode(user_query)
    return query_embedding

# @tool
def instance_extractor_tool(state: InfoExtractionState):
    """Tool to identify relevant classes according to a user query."""

    identified_class = state["identified_class"]
    query = state["query"][-1]
    reasoning_clues = state["reasoning_clues"]
    
    class_sel_name = list(identified_class.keys())

    for i, name in enumerate(class_sel_name):
        if name not in list(instance_dict.keys()):
            class_name = class_match(list(instance_dict.keys()), name)
            class_sel_name[i] = class_name

    relevant_instances = []
    # query_embedding = embed_query(list(query) + list(tasks))
    query_embedding = embed_query(", ".join(query) + ", ".join(reasoning_clues))
    
    for item in class_sel_name:
        if item not in list(instance_dict.keys()):
            continue
        if len(instance_dict[item]) >100:
            df_top_chunks = calculate_similarity(instance_dict_emb[item], query_embedding, top_k=100)
            elements = [instance_dict[item][i] for i in df_top_chunks.index.tolist()]
            relevant_instances.append(elements)
        else:
            relevant_instances.append(instance_dict[item])

    if len(relevant_instances) > 0:
        relevant_instances = [item for sublist in relevant_instances for item in sublist]

    return {"relevant_instances": relevant_instances}

instance_ext_graph = StateGraph(InfoExtractionState)
instance_ext_graph.add_node("instance_extracting_tools", instance_extractor_tool)
instance_ext_graph.set_entry_point("instance_extracting_tools")
instance_ext_graph.set_finish_point("instance_extracting_tools")
app3 = instance_ext_graph.compile()
# cc = app3.invoke(bb, stream_mode="values")

In [ ]:

def get_explanation(class_uri):
    
    sparql.setReturnFormat(JSON)
    query = f"""
    SELECT ?comment WHERE {{
        <{class_uri}> rdfs:comment ?comment .
        FILTER (lang(?comment) = "en")
    }} LIMIT 1
    """
    sparql.setQuery(query)
    try:
        results = sparql.query().convert()
        bindings = results["results"]["bindings"]
        if bindings:
            extracted_info = bindings[0]["comment"]["value"]
        else:
            print("No explanation available.")
            extracted_info = 'Not available.'
    except Exception as e:
        print(f"Error: {str(e)}")
        extracted_info = 'Not available.'
        
    return extracted_info

#   For each instance, try to get its comment
def get_instance_explanation(relevant_instances: list):
    comments_collection = []
    for item in relevant_instances:
        comment = get_explanation(item)
        # print(comment)
        combined = str(item) + str(comment)
        comments_collection.append(combined)

        return comments_collection

def select_instance_llm(relevant_instances: list, query: str, comments_collection: list, reasoning_clues) -> dict:
    
    llm7 = ChatOpenAI(model="o3")
    example_output = {'air_pressure_at_convective_cloud_top': 'https://w3id.org/hacid/data/cs/variable/cf/air_pressure_at_convective_cloud_top'}
    messages = [
        SystemMessage(content=f"""Your are instance selector with expertise in climate services and climate data. \
                      Given a list of instances represented in URI {relevant_instances}, \
                      Your job is to select the most useful instances from the above list to answer the user query about climate services, i.e. {query}. \
                      Some guidances are provided to help you with the selection, i.e. {reasoning_clues}.\
                      The maximum number of instances to be selected is 20. \
                      The minimum number of instance to be selected is 1. \
                      The output should be a dictionary which can contain multiple items with the instance name as key and the class URI as value, as shown in the example: {example_output}. \
                      The instance name is the last segment of the URI. \
                      Return only the dictionary containing the instance name and its uri in the output. No additional information. \
                      """),
        HumanMessage(content=[{"type":"text", "text":f"Query: {query}"}])]

    response = llm7.invoke(messages)
    
    # return response
    instance_dict = ast.literal_eval(response.content)
    return instance_dict


def select_instance_human(instance_list):
    # Create a mapping from suffix to full URL
    suffix_map = {}
    for url in instance_list:
        suffix = url.rstrip('/').split('/')[-1]
        suffix_map[suffix.lower()] = url  # store lowercase for easy matching
    
    # Display available suffixes
    print("Available models:")
    for suffix in suffix_map:
        print(suffix)

    while True:
        user_input = input("Type the model names separated by commas, or 'all' to select all: ").strip().lower()
        if user_input == 'all':
            # Select all models
            selected_urls = list(suffix_map.values())
            break
        else:
            model_names = [name.strip() for name in user_input.split(',')]
            invalid_models = [name for name in model_names if name not in suffix_map]
            if invalid_models:
                print(f"Models not found: {', '.join(invalid_models)}. Please try again.")
            else:
                selected_urls = [suffix_map[name] for name in model_names]
                break
    return selected_urls


# def instance_selector_tool(relevant_instances, task_list, query, use_llm) -> dict:
def instance_selector_tool(state: InfoExtractionState):
    """Identify the most relevant instances within each class."""
    relevant_instances = state["relevant_instances"]
    query = state["query"][-1]
    reasoning_clues = state["reasoning_clues"]
    comments_collection = get_instance_explanation(relevant_instances)
    use_llm = True
    if use_llm:
        recommended_instance = select_instance_llm(relevant_instances, query, comments_collection, reasoning_clues)
        selected_instances = [item.strip() for item in list(recommended_instance.values())]
    else:
        ### User Confirmation ###
        selected_instances = select_instance_human(relevant_instances)
    print(f"Selected instances: {selected_instances}")
    
    return {"selected_instances": selected_instances}


instance_sel_graph = StateGraph(InfoExtractionState)
instance_sel_graph.add_node("instance_selecting_tools", instance_selector_tool)
instance_sel_graph.set_entry_point("instance_selecting_tools")
instance_sel_graph.set_finish_point("instance_selecting_tools")
app4 = instance_sel_graph.compile()
# dd = app4.invoke(cc, stream_mode="values")

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

def extract_relevant_entities(target_uri: str):
    """
    Extract entities and their relationships.
    """

    # sparql_endpoint = "https://keng.istc.cnr.it/hacid-cs-kg/sparql"
    sparql_endpoint = "https://keng.istc.cnr.it/hacid-cs-kg/sparql"

    sparql = SPARQLWrapper(sparql_endpoint)
    sparql.setReturnFormat(JSON)
    
    # Query to get all predicate-object triples where the URI is the subject
    select_query = f"""
    SELECT ?p ?o
    WHERE {{
      <{target_uri}> ?p ?o .
    }}
    """
    sparql.setQuery(select_query)

    try:
        results = sparql.query().convert()

        triples = []
        for res in results["results"]["bindings"]:

            pred = res["p"]["value"]
            obj = res["o"]["value"]
            triples.append({"predicate": pred, "object": obj})

    except Exception as e:
        print(f"Error during SPARQL query: {e}")
        triples = []
    
    # Organize triples into a dictionary: predicate URI -> list of objects
    info_dict = {}
    if not triples:
        print(f"No information found for {target_uri}")
        return {}
    else:
        for triple in triples:
            predicate = triple["predicate"]
            obj = triple["object"]
            if predicate not in info_dict:
                info_dict[predicate] = []
            info_dict[predicate].append(obj)

    return info_dict

def embed_query(user_query):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    # model = SentenceTransformer('all-distilroberta-v1')
    query_embedding = model.encode(user_query)
    return query_embedding


def subgraph_extractor_tool(state: InfoExtractionState):    
    """Extract the sub-graph for each instance."""
    selected_instances = state["selected_instances"]
    query = state["query"][-1]
    reasoning_clues = state["reasoning_clues"]

    relevant_info = {}
    for item in selected_instances:
        entities = extract_relevant_entities(item)
        for key, value in entities.items():
            if key not in relevant_info:
                relevant_info[key] = []
            if value not in relevant_info[key]:
                relevant_info[key].append(value)

    query_embedding = embed_query(", ".join(query) + ", ".join(reasoning_clues))

    combined_info = {k: sum(v, []) for k, v in relevant_info.items()}
    for k, v in combined_info.items():
        if len(v) > 20:
            # combined_info[k] = random.sample(v, 20)
            instance_subset = instance_embeddings_all[instance_embeddings_all['instance_uri'].isin(combined_info[k])]
            # instance_subset['instance_emb'] = instance_subset['instance_emb'].apply(lambda x: np.array(x))
            instance_subset.loc[:, 'instance_emb'] = instance_subset['instance_emb'].apply(lambda x: np.array(x))
            array = instance_subset['instance_emb'].values
            df_top_association = calculate_similarity(array, query_embedding, top_k=20)
            elements = [instance_subset.loc[instance_subset.index.tolist()[i], 'instance_uri'] for i in df_top_association.index.tolist()] 
            combined_info[k] = elements

    subgraph_instances_col = list(combined_info.values())
    subgraph_instances = [item for sublist in subgraph_instances_col for item in sublist]
    if len(subgraph_instances) > 20:
        subgraph_embedding = instance_embeddings_all[instance_embeddings_all['instance_uri'].isin(subgraph_instances)]
        subgraph_embedding.loc[:, 'instance_emb'] = subgraph_embedding['instance_emb'].apply(lambda x: np.array(x))
        array_subgraph = subgraph_embedding['instance_emb'].values
        df_top_association_sel = calculate_similarity(array_subgraph, query_embedding, top_k=20)
        elements_sel = [subgraph_embedding.loc[subgraph_embedding.index.tolist()[i], 'instance_uri'] for i in df_top_association_sel.index.tolist()] 
        combined_info_sel = elements_sel        
    else:
        combined_info_sel = subgraph_instances

    print(f"Relevant info from KG: {combined_info_sel}")
    
    return {"subgraphs": combined_info_sel}

instance_subgraph_ext_graph = StateGraph(InfoExtractionState)
instance_subgraph_ext_graph.add_node("subgraph_extractor_tool", subgraph_extractor_tool)
instance_subgraph_ext_graph.set_entry_point("subgraph_extractor_tool")
instance_subgraph_ext_graph.set_finish_point("subgraph_extractor_tool")
app5 = instance_subgraph_ext_graph.compile()
# ee = app5.invoke(dd, stream_mode="values")

In [ ]:

class ResponseGenerationState(TypedDict):
    messages: list
    query: str
    project_description: str
    identified_class: list
    selected_instances: list
    subgraphs: list
    responses: list
    reasoning_clues: list


def contex_select(previous_context, query):
    """Function to identify relevant context from previous answers."""

    # llm2 = ChatOpenAI(model="o1")
    llm2 = ChatOpenAI(model="gpt-4o")
    example_output = {'Context': 'Summarised text of relevant context from previous answers'}
    messages = [
        SystemMessage(content=f"""Your are an expert in discovering dependencies in the climate services workflow. \
                      Given a user query and the initial plan, your job is to analyse and identify the relevant information from the previous query answers. \
                
                      Follow the tips below: \
                        1. Context: Summarise and understand the context from the previous answers: {previous_context}. \
                        2. Current query: Understand the current user query based on the query text: {query} \
                        3. Dependency: Think and identify the dependencies from the previous answers relevant to the current user query, including useful information, constriants, and conditions. \
                        4. Extract: Extract relevant contexts and summarise concisely. \
                      
                      The output should be a dictionary with the Context as the key and the summarised text as value, as shown in the example: {example_output}. \
                      Return only the dictionary. No additional information. \
                      """),
        HumanMessage(content=[{"type":"text", "text":f"Query : {query}"}])]

    if len(previous_context)==0:
        answer_context = {'Context': 'Not Existing'}
        print(f"Context from the previous answers does not exist.")
    else:
        response = llm2.invoke(messages)
        start_idx = response.content.find('{')
        end_idx = response.content.rfind('}') + 1
        answer_context = response.content[start_idx:end_idx]
        # answer_context = ast.literal_eval(response.content)
        print(f"Context from the previous answers: {answer_context}")
    # return {"identified_class": class_dict}
    return answer_context
   

# def response_generator_tool(user_query, project_info, identified_class, selected_instances, relevant_info):
def response_generator_tool(state: ResponseGenerationState):
    """Generate a response based on the extracted graph information and user query."""

    user_query = state["query"][-1]
    project_info = state["project_description"]
    identified_class = state["identified_class"]
    selected_instances = state["selected_instances"]
    relevant_info = state["subgraphs"]
    previous_context = state["responses"]
    reasoning_clues = state["reasoning_clues"]

    selected_instances_explanations = get_instance_explanation(selected_instances)
    previous_context = [json.loads(item).get('Answer') for item in previous_context]
    answer_context = contex_select(previous_context, user_query)

    class_sel_name = list(identified_class.keys())
    class_sel_uri = [identified_class[item].strip() for item in class_sel_name]
    class_sel_explanation = [class_info[item] for item in class_sel_name if item in class_info]
    
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a climate services expert. "
                    "Your responsibility is to answer queries related to climate services according to the provided relevant context information. "
                    "For a given query, you will be provided with the user query, project description, historical answers context, relevant context for the current query retrieved from knowledge graph as below." 
                    f"1. User query: {user_query} "   
                    f"2. Project information: {project_info} "
                    f"3. Context from previous answers: {answer_context} \n\n"
                    f"4. Relevant context from knowledge graph: Class name: {class_sel_name}, Class explanation: {class_sel_explanation}, Selected instance within the class: {selected_instances} and their explanations {selected_instances_explanations}, and Relevant nodes and relationships: {relevant_info} \n\n"

                    "Answer the query based on the **retrieved relevant context** from the knowledge graph as provided above, refine it according to historical context and project description."
                    "If no context information is retrieved, just state no relevant infomation from the knowledge graph."
                    "You need to be concise and specific in your answers."
                    "Frame your response **strictly in valid JSON format** as shown below. Do not include any additional text, explanations, or markdown—only output the JSON object."
                    "Here is the JSON structure you should follow:\n"
                    '{\n'
                    f' "Answer": "<Provide your answer for the query here.>",\n'
                    '  "Context from knowledge graph": "<Translate, explain, and summarise the retrieved context information from knowledge graph as above. >",\n'
                    '  "Further reasoning": "<any additional reasoning or information>"\n'
                    '}'
                )
            },
            {
                "role": "user",
                "content": f"Query : {user_query}"
            }
        ]
    )

    responses = response.choices[0].message.content
    return {"responses": responses}

response_generation_graph = StateGraph(ResponseGenerationState)
response_generation_graph.add_node("response_generator_tool", response_generator_tool)
response_generation_graph.set_entry_point("response_generator_tool")
response_generation_graph.set_finish_point("response_generator_tool")
app6 = response_generation_graph.compile()


In [ ]:
from typing import Literal
from langchain_core.language_models.chat_models import BaseChatModel
from typing_extensions import TypedDict
from langgraph.graph import MessagesState, START, END
from langgraph.types import Command


class OverallState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    query: list
    project_description: str
    reasoning_clues: list
    identified_class: list
    relevant_instances: list
    selected_instances: list
    subgraphs: list
    responses: list


def make_supervisor_node(llm: BaseChatModel, members: list[str]) -> str:
    members = ["class_identifier","instance_extractor", 
                "instance_selector", "subgraph_extractor", "answer_generator"]
    options = members + ["FINISH"]

    system_prompt = (
        f"You are a supervisor tasked with solving a climate services case provided by the user."
        "You are equipped with the following workers: {members}, to help you solve different sub-tasks." 
        "Follow the instructions as below:"
        "1. You must follow the sequence: firstly class_identifier, secondly instance_extractor, thirdly instance_selector, then subgraph_extractor, finally answer_generator."
        "2. You must start with class_identifier, follow the sequence. After answer_generator is run, you should route to FINISH."
        "3. For each query, you should **only run the above sequence once**. Do not skip or repeat running any worker. You must not repeat running any worker more than once."
        "4. Only call class_identifier, instance_extractor, instance_selector, subgraph_extractor, and answer_generator once."
        "5. Make sure you have run answer_generator. If not, you must route to answer_generator."
        "6. When finished, respond with FINISH."
    )

    class Router(TypedDict):
        """Worker to route to next. If no workers needed, route to FINISH."""

        next: Literal["class_identifier","instance_extractor", 
                      "instance_selector", "subgraph_extractor", "answer_generator", "FINISH"]

    def supervisor_node(state: OverallState) -> Command[Literal["class_identifier","instance_extractor", 
                                                                "instance_selector", "subgraph_extractor", "answer_generator", "__end__"]]:
        print("updatedstate", state)
        messages = [
            {"role": "system", "content": system_prompt},
        ] + state["messages"]
        
        response = llm.with_structured_output(Router).invoke(messages)
        goto = response["next"]
        print(f"Next Worker: {goto}")
        if goto == "FINISH":
            goto = END

        return Command(goto=goto, update={"next": goto})
    
    return supervisor_node

supervisor_node = make_supervisor_node(llm, ["class_identifier","instance_extractor", "instance_selector", "subgraph_extractor", "answer_generator"])


def class_identification_node(state: InfoExtractionState) -> Command[Literal["supervisor"]]:
    class_result = app2.invoke(state)
    return Command(
        update={
            "messages": [
                HumanMessage(content=class_result["messages"][-1].content, name="class_identifier")
            ],
            "identified_class": class_result["identified_class"],
        },
        goto="supervisor",
    )

def instance_extraction_node(state: InfoExtractionState) -> Command[Literal["supervisor"]]:
    instance_extraction_result = app3.invoke(state)
    return Command(
        update={
            "messages": [
                HumanMessage(content=instance_extraction_result["messages"][-1].content, name="instance_extractor")
            ],
            "relevant_instances": instance_extraction_result["relevant_instances"],
        },
        goto="supervisor",
    )

def instance_selection_node(state: InfoExtractionState) -> Command[Literal["supervisor"]]:
    instance_selection_result = app4.invoke(state)
    return Command(
        update={
            "messages": [
                HumanMessage(content=instance_selection_result["messages"][-1].content, name="instance_selector")
            ],
            "selected_instances": instance_selection_result["selected_instances"],
        },
        goto="supervisor",
    )

def subgraph_extraction_node(state: InfoExtractionState) -> Command[Literal["supervisor"]]:
    subgraph_extraction_result = app5.invoke(state)
    return Command(
        update={
            "messages": [
                HumanMessage(content=subgraph_extraction_result["messages"][-1].content, name="subgraph_extractor")
            ],
            "subgraphs": subgraph_extraction_result["subgraphs"],
        },
        goto="supervisor",
    )

def response_generator_node(state: ResponseGenerationState) -> Command[Literal["supervisor"]]:
    response_result = app6.invoke(state)
    return Command(
        update={
            "messages": [
                HumanMessage(content=response_result["messages"][-1].content, name="answer_generator"), 
            ],
            "responses": response_result["responses"],
        },
        goto="supervisor",
    )

parent_graph_builder = StateGraph(OverallState)
parent_graph_builder.add_node("supervisor", supervisor_node)
parent_graph_builder.add_node("class_identifier", class_identification_node)
parent_graph_builder.add_node("instance_extractor", instance_extraction_node)
parent_graph_builder.add_node("instance_selector", instance_selection_node)
parent_graph_builder.add_node("subgraph_extractor", subgraph_extraction_node)
parent_graph_builder.add_node("answer_generator", response_generator_node)

parent_graph_builder.add_edge(START, "supervisor")
parent_graph = parent_graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(parent_graph.get_graph().draw_mermaid_png()))

In [ ]:

def base_recipe_tool(query_template, project_description):
    """Tool to create a base recipe for a climate services case."""

    llm8 = ChatOpenAI(model="gpt-5")
    # llm8 = ChatOpenAI(model="gpt-4o")
    example_output = [{'Recipe - Q1': 'Answer for Q1'},
                        {'Recipe - Q2': 'Answer for Q2'},
                        {'Recipe - Q3': 'Answer for Q3'}]
    messages = [
        SystemMessage(content=f"""Your are a climate service expert. Your job is to create bespoke climate services recipes based on project description. \
                      You will be provided with the background information of a climate service project {project_description} and a template of queries {query_template} for generating recipes step by step. \
                      You need to read the project description to understand the case, the user requirements, and priority.  \
                      You then need to read the query template to understand what key information are needed and the process of creating a wholistic climate services recipe. \
                      You then create a whole recipe with consistency for the client based on your internal knowledge and reasoning capability.
                      You are allowed to provide multiple recommendations in absence of clear context from the client, you have to ensure the consistency of the recipe from start to end. \
                      The output should be a dictionary, as shown in the example: {example_output}. \
                      Each dictionary should contain expected outcomes for the query and key steps to answer the query. Each answer should be less than 250 words long.\
                      Be concise and consistent. Prioratise the most important information. Do not hallucinate.\
                      Return only the list of dictionaries. No additional information. \
                      """),
        HumanMessage(content=[{"type":"text", "text":f"Query : {query_template}"}])]

    response = llm8.invoke(messages)
    base_recipe = ast.literal_eval(response.content)

    # start_idx = base_recipe.find('{')
    # end_idx = base_recipe.rfind('}') + 1
    # base_recipe = base_recipe[start_idx:end_idx]
    # base_recipe = ast.literal_eval(content)
    

    return base_recipe


In [ ]:

def recipe_aggregation_tool(query, project_description, agentic_recipe, base_recipe):
    """Aggregate base recipes and agentic recipes."""

    
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a senior climate services expert."
                    "You will be provided with the following information."
                    f"Project description: {project_description}"
                    f"Query: {query}\n\n"
                    f"Agentic Recipe: {agentic_recipe}\n\n"
                    f"**Base Recipe**: {base_recipe}\n\n"
                    "Your responsibility is to **critically review and select** the most reasonable information from the agentic recipes and base recipes and combine those information to form the finalised climate services recipe."
                    "Some tips to follow:" 
                    "1. Read the project description to understand the requirements and priority of the client."
                    "2. Reflect on the quality of both agentic and base recipes against the client requirements."
                    "3. Compare the two recipes and identify the differences and similarities."
                    "4. Decide the relationship from the two recipes, i.e. conflicting or complimentary."
                    "5. If they are conflicting, prioratise the information from the <Base Recipe>."
                    "6. If they are complimentary, combine the information from both recipes to form a more comprehensive recipe."
                    "7. Use the <Base Recipe> as your first option for the key information about emission scenarios, historical reference period and specific climate models."
                    "8. Use Agentic Recipe to augment the Base Recipe"
                    
                    "Only include information that directly answers the question in the query."
                    "If no context info from HACID Knowledge Graph then do not use agentic recipes for aggregation."
                    "Then create a coherent, logical, and concise answer. The answer should start with the most important information, followed by secondary information."
                    "You must be concise with your language and only keep most useful information that directly answers the query."
                    "Answer each query one by one. **Do not skip any query**."
                    "Frame your response **strictly in valid JSON format** as shown below. Do **not** include any additional text, explanations, or markdown—only output the JSON object."
                    "Here is the JSON structure you should follow:\n"
                    '{\n'
                    '  "Aggregated answer": "<your answer here based on the agentic recipe and base recipe>",\n'
                    '  "Context from knowledge graph": "<context information contained in the agentic recipe.>",\n'
                    '  "Context from LLM reasoning": "<context information provided by the base recipe>",\n'
                    '  "Evaluation": "<You evaluation using a scale from 1 to 5 (1 means poor and 5 means great) on four dimensions, i.e. comprehensive, specific, accurate, and concise.  "Base Recipe - Comprehensive: , Specific: , Accurate: , Concise: ; Agentic Recipe - Comprehensive: , Specific: , Accurate: , Concise: ; Aggregated Answer - Comprehensive: , Specific: , Accurate: , Concise: ". >"\n'
                    '}'
                )
            },
            {
                "role": "user",
                "content": f"Query : {query}"
            }
        ]
    )

    responses = response.choices[0].message.content
    start_idx = responses.find('{')
    end_idx = responses.rfind('}') + 1
    aggregated_answer = responses[start_idx:end_idx]
    return aggregated_answer


def refine_aggregation(query, answer, aggregated_answers):
    """Refine the aggregated answer."""

    previous_context = [json.loads(item).get('Refined Answer') for item in aggregated_answers]
    answer_context = contex_select(previous_context, query)

    answer = json.loads(answer).get('Aggregated answer')
    
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a senior climate services expert."
                    "Your responsibility is to refine the answer using the context from the previous answers to make it more consistent and coherent."
                    "You will be provided with the following information."
                    f"The current query: {query}\n\n"
                    f"The answer for the current query: {answer}\n\n"
                    f"The relevant context extracted from previous answers: {answer_context}\n\n"
                    "Some actions to follow: "
                    "1. For each question in the list, read the question and its current answer"
                    "2. Review and evaluate if answer context is directly linked to the current query but not captured by the current answer."
                    "3. Reflect if there are conflicts between the current answer and the answer context. Resolve the conflict to ensure consistency accoring to the project information."
                    "4  Refine the current answer according to the reflection. If answer context not directly linked, just keep the current answer as is."
                    "Do not introduce any new information that is not present in either the current answer or the answer context."
                    "If there is nothing relevant to add, just return the current answer as is."
                    "Remove redundant information and ensure the answer is concise and to the point, and coherent."
                    "Frame your response **strictly in valid JSON format** as shown below. Do **not** include any additional text, explanations, or markdown—only output the JSON object. Do **not** include query in your answer."
                    "Here is the JSON structure you should follow:\n"
                    '{\n'
                    '  "Refined answer": "<your answer here>",\n'
                    '  "Evaluation": "<You evaluation using a scale from 1 to 5 (1 means poor and 5 means great) on four dimensions, i.e. comprehensive, specific, accurate, and concise. Refined Answer - Comprehensive: , Specific: , Accurate: , Concise: ". >"\n'
                    '}'
                )
            },
            {
                "role": "user",
                "content": f"Query : {query}"
            }
        ]
    )

    responses = response.choices[0].message.content
    start_idx = responses.find('{')
    end_idx = responses.rfind('}') + 1
    refined_aggregated_answer = responses[start_idx:end_idx]
    
    return  refined_aggregated_answer


In [ ]:
case = 'Water Demand - batch 2'
import pandas as pd
queries = pd.read_csv(os.path.join(f"HACID DSS Evaluation/Case Results/{case}", f"{case}.csv"), encoding='Windows-1252')
query_template = queries["Query"]
queries["States"] = queries["States"].astype(object)
project_description = '''
High-level Summary: Change in future water demand in an anonymous water resource planning region in the UK as a result of climate change.
Key Question: How will the frequency and duration of peak water demand events, and the water demand associated with key percentiles, change in the future compared with the current climate?
Region: One of the water resource planning regions in the UK
Hazard: Heatwaves
Impact: Longer-duration and more frequent heatwaves cause sustained and more frequent increases in water demand. Higher temperatures (increase in the 90th and 95th percentile) can contribute to an increased amount of water demanded by customers.
Exposure: Water demand by exposed customers
Vulnerability: Heatwaves (especially combined with drought) cause an increase in water demand due to higher consumption for personal use, leisure, and more often garden watering.
Decision Context: Better water resource management to ensure supply meets demand even at peak times and for the duration of peaks; customer education about the supply challenges associated with increased demand; improved information provision to the regulators and licensors such as Ofwat and the Environment Agency.
Risk Tolerance: Medium
Time Horizon: 
Warming/Emission Scenarios: Customer is interested in global warming levels of 1.5°C, 2°C, 3°C, and 3.5°C.
Spatial Resolution:  Sufficient to resolve water resource planning regions
Temporal Resolution: At least daily
Uncertainty: There is also a requirement to quantify uncertainty.
Additional customer information: 
'''
base_recipe = base_recipe_tool(query_template, project_description)

overall_state = OverallState()
overall_state["project_description"] = project_description
overall_state["tasks"] = []
overall_state["identified_class"] = []
overall_state["relevant_instances"] = []
overall_state["selected_instances"] = []
overall_state["subgraphs"] = []
overall_state["responses"] = []

answers = []

for i in range(len(queries)):
    print(f"Processing query: {i+1} - {queries.loc[i, 'Query']}")
    overall_state["query"] = [queries.loc[i, 'Query']]
    overall_state["messages"] = list(overall_state["query"] + [overall_state["project_description"]])
    overall_state["reasoning_clues"] = base_recipe[i]
    overall_state['responses'] = answers
    # print(f"*** The updated response is: {overall_state['responses']} ! ***")
    gg = parent_graph.invoke(overall_state, stream_mode="updates")
    answer = gg[-2]['answer_generator']['responses']
    start_idx = answer.find('{')
    end_idx = answer.rfind('}') + 1
    answer = answer[start_idx:end_idx]
    print(f"Answer for query {i+1}: {answer}")
    answers.append(answer)
    queries.loc[i, 'Answer'] = answer
    queries.at[i, "States"] = gg

with open(f'HACID DSS Evaluation/Case Results/{case}/{case} Agentic CS recipe.pkl', 'wb') as file:
    pickle.dump(queries, file)
queries_output = queries[['Query', 'Type', 'Answer']]   
queries_output.to_csv(f"HACID DSS Evaluation/Case Results/{case}/{case} recipe.csv", index=False)
gg = parent_graph.invoke(overall_state, stream_mode="updates")

aggregated_answers = []
aggregated_answers_refined = []
for i in range(len(queries)):
    aggregated_answer = recipe_aggregation_tool(queries.loc[i, 'Query'], project_description, answers[i], base_recipe[i])
    aggregated_answers.append(aggregated_answer)
    refined_aggregated_answer = refine_aggregation(queries.loc[i, 'Query'], aggregated_answer, aggregated_answers_refined)
    aggregated_answers_refined.append(refined_aggregated_answer)

answers_collection = [queries["Query"].tolist(), base_recipe, answers]
df_answers = pd.DataFrame({'Query': queries["Query"].tolist(), 'Reasoning': base_recipe, 'Agentic': answers, 'Aggregated original':aggregated_answers, 'Aggregated refined':aggregated_answers_refined})
df_answers.to_csv(f"HACID DSS Evaluation/Case Results/{case}/{case} CS recipe aggregated.csv", index=False)


In [ ]:
file_path = "HACID Evaluation/Case Results/Finalised results/Finalised HACID Results - cleaned - v2.xlsx"
# cleaned_recipes = pd.read_excel(file_path, sheet_name='Sheet1')
BE_recipe = pd.read_excel(file_path, sheet_name='Built Environment')
EN_recipe = pd.read_excel(file_path, sheet_name='Energy')
TR_recipe = pd.read_excel(file_path, sheet_name='Transport')
AG_recipe = pd.read_excel(file_path, sheet_name='Agriculture')
SLR_recipe = pd.read_excel(file_path, sheet_name='Sea Level Rise')


In [ ]:

def streamline_recipe_tool(recipes, project_description):
    """Tool to create a base recipe for a climate services case."""
    recipes = ','.join(recipes['Aggregated original'].astype(str))

    llm8 = ChatOpenAI(model="gpt-4o")
    # llm8 = ChatOpenAI(model="gpt-4o")
    example_output = [{'Climate Projection': 'A specfic climate projection data product'},
                        {'Spatial Resolution': 'The spatial resolution'},
                        {'Temporal Resolution': 'The temporal resolution'},
                        {'Warming/Emission Scenarios': 'Scenarios required'},
                        {'Climate Variables': 'Essential climate variables needed'},
                        {'Historical Observation': 'A specfic historical observation data product'},
                        {'Data Preprocessing': 'Preprocessing required'},
                        {'Essential Calculations': 'Calculations required'},
                        {'Derived Metrics': 'Metrics to be derived'},
                        {'Results Presentation': 'Postprocessing required'}]
    
    messages = [
        SystemMessage(content=f"""Your are a climate service expert. Your job is to create a practical and consistent solution regarding climate services for a given project. \
                      You will be provided with the background information of a climate service project {project_description} and a predefined recipes {recipes}. \
                      You will conclude on the key information as below, by **extracting specific recommendations available from the predefined recipes**.\
                      1. **Climate Projection**: Which climate projection data product to use ? \
                      2. **Spatial Resolution**: at what spatial resolution ? \
                      3. **Temporal Resolution**: at what temporal resolution ? \
                      4. **Warming/Emission Scenarios**: What global warming/emission scenarios to use ? \
                      5. **Climate Variables**: What specific climate or meteorological variables to use ? \
                      6. **Historical Observation**: Which historical observation data product to use ? \
                      7. **Data Preprocessing**: What data preprocessing methods are involved ? \
                      8. **Essential Calculations**: What essential calculations are involved to inform the climate services case ? \
                      9. **Derived metrics**: What metrics should be derived to inform the case ? \
                      10. **Results Presentation**: What types of presentation should be utilised ? \
                      
                      All the inofrmation should be extracted from the predefined recipes. Do not introduce new information.\
                      When there are inconsistencies among the predefined recipes or multiple solutions in the recipes, you need to use your expertise to select the most reasonable information to ensure the consistency of the final recipe. \
                      Be concise and specific. **You have to be consistent**.\

                      The output should be a dictionary, as shown in the example: {example_output}. \

                      Return only the list of dictionaries. No additional information. \
                      """),
        HumanMessage(content=[{"type":"text", "text":f"Query : {query_template}"}])]

    response = llm8.invoke(messages)
    streamlined_recipe = ast.literal_eval(response.content)
    
    return streamlined_recipe


In [ ]:
project_description = '''
High-level Summary: Climate projections for coastal inundation at a sea rescue charity’s asset along the UK coastline
Key Question: How will future coastal flooding affect the customer’s assets along the coastline?
Region: UK coastline
Hazard: Extreme sea level, sea level rise
Impact: Sea level rise might submerge or cause the locations where the customer's infrastructure is located to be below the highest astronomical tide. 
Coastal flooding might damage the infrastructure through the increase in return levels beyond what was the design of the building.
Exposure: Lifeboat stations, customer's training centre buildings, lifeguard stations
Vulnerability: The basement level of buildings usually provides some margin for sea level increase, but this may not be sufficient to accommodate a higher tidal range, increased mean sea level, and additional storm surge combined. The charity’s training centre appears to be above a coastal protection structure, which likely has enough additional height to withstand extreme sea levels. However, there is limited opportunity to adapt to sea level rise other than by rebuilding infrastructure—a challenging prospect for a charity with constrained funding.
Decision Context: The optimal height over datum for any new stations built in each coastal area along the coast; prioritisation of which stations need adaptations or rebuilding to cope with coastal floods of the future.
Risk Tolerance: The project has a medium risk tolerance and requires high fidelity in the assessment. 
Time Horizon: 01/01/2030 to 1/12/2130
Warming/Emission Scenarios: Considering various climate scenarios for coastal inundation and their potential impacts on coastal buildings.
Spatial Resolution:  Climate data used for the assessment should be as close as possible to the asset locations along the coastline (250 lifeboat and 250 lifeguard stations along the coastline).
Temporal Resolution: The temporal resolution should be annual or finer. 
Uncertainty: There is also a requirement to quantify uncertainty.
Additional customer information: The customer can provide an inventory of all assets, including building archetypes and land survey information.
'''


In [ ]:
# BE_streamlined = streamline_recipe_tool(BE_recipe, project_description)
# BE_streamlined
# EN_streamlined = streamline_recipe_tool(EN_recipe, project_description)
# EN_streamlined
# TR_streamlined = streamline_recipe_tool(TR_recipe, project_description)
# TR_streamlined
# AG_streamlined = streamline_recipe_tool(AG_recipe, project_description)
# AG_streamlined
SLR_streamlined = streamline_recipe_tool(SLR_recipe, project_description)
SLR_streamlined